# L76 · 音乐理论速成 — 音高、音程、色度轮与十二平均律

**学习目标**
- 理解 12 个音高类别（pitch class）和八度等价性
- 掌握 MIDI ↔ 频率转换公式
- 实现色度轮可视化
- 明白为什么 audio AI 用 chroma 而非原始频率

## 音乐的基本单元：音高类别

人耳把频率每翻一倍（一个八度）感知为"同名音"：
130Hz C2、261Hz C3、523Hz C4 在和声上是等价的。

**十二平均律（12-EDO）** 把一个八度等分成 12 个半音：

| 编号 | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| 音名 | C | C# | D | D# | E | F | F# | G | G# | A | A# | B |

**MIDI 公式**：`f(n) = 440 × 2^((n-69)/12)`，其中 n 是 MIDI 编号（A4=440Hz=MIDI69）

**色度 = MIDI mod 12**：忽略八度，只保留音高类别。这是音乐 AI 的核心特征。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

PITCH_CLASSES = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
MIDI_A4, FREQ_A4 = 69, 440.0

## ✏️ 任务 1：MIDI ↔ 频率转换

In [ ]:
def midi_to_freq(midi: float) -> float:
    """MIDI 编号 → 频率 Hz。A4=69=440Hz。"""
    # ✏️ TODO: f(n) = 440 × 2^((n-69)/12)
    return FREQ_A4 * 2 ** ((midi - MIDI_A4) / 12.0)

def freq_to_midi(freq: float) -> float:
    """频率 Hz → 浮点 MIDI 编号。"""
    # ✏️ TODO: 逆公式：n = 69 + 12 × log2(f / 440)
    return MIDI_A4 + 12.0 * np.log2(max(freq, 1e-8) / FREQ_A4)

# 验证
for name, midi, expected in [('A4',69,440.0),('C4',60,261.63),('A3',57,220.0)]:
    got = midi_to_freq(midi)
    ok = abs(got - expected) < 0.5
    print(f'{name} MIDI{midi}: {got:.2f} Hz  {"✅" if ok else "❌"}')

assert abs(freq_to_midi(440.0) - 69) < 0.01
print('✅ 频率转换验证通过')

## ✏️ 任务 2：chroma_from_freq

In [ ]:
def chroma_from_freq(freq: float) -> int:
    """频率 Hz → 音高类别 0-11 (C=0, C#=1, ..., B=11)。"""
    # ✏️ TODO: 用 freq_to_midi 换算后取 mod 12
    return int(round(freq_to_midi(freq))) % 12

# 验证
for name, freq, expected_pc in [('C4',261.63,0),('A4',440.0,9),('A5',880.0,9),('B4',493.88,11)]:
    pc = chroma_from_freq(freq)
    print(f'{name} ({freq:.1f}Hz): chroma={pc} ({PITCH_CLASSES[pc]}) {"✅" if pc==expected_pc else "❌"}')

In [ ]:
# 🎨 色度轮可视化（教学图表，aviz 不涵盖此内容）
fig, ax = plt.subplots(figsize=(5, 5), subplot_kw={'projection': 'polar'})
angles = np.linspace(0, 2*np.pi, 12, endpoint=False)
colors = plt.cm.hsv(np.linspace(0, 1, 12, endpoint=False))
ax.bar(angles, np.ones(12), width=2*np.pi/12, color=colors, alpha=0.75, align='edge')
for i, (a, name) in enumerate(zip(angles + np.pi/12, PITCH_CLASSES)):
    ax.text(a, 1.35, name, ha='center', va='center', fontsize=10, fontweight='bold')
ax.set_yticklabels([])
ax.set_xticklabels([])
ax.set_title('色度轮 — 12 音高类别', pad=15)
plt.tight_layout()
plt.savefig('/Users/z/AURORA/notebooks/8_music/chroma_wheel.png', dpi=80, bbox_inches='tight')
plt.show()

## 大调音阶与 chroma

C 大调 = C-D-E-F-G-A-B = chroma {0,2,4,5,7,9,11}。

同一首曲子升调后（转调），整个 chroma 向量旋转。
不同调性的歌曲，chroma 分布形状不同但"模式"相似。

这就是为什么 chroma 向量能做**调性无关的**音乐相似度比较。

## 小结

| 概念 | 要记住 |
|---|---|
| 12-EDO | 八度=12半音，相邻半音频率比 = 2^(1/12) ≈ 1.0595 |
| MIDI | f(n)=440×2^((n-69)/12)，A4=MIDI69=440Hz |
| Chroma | MIDI mod 12，消除八度，保留音高类别 |
| L77 | 把 chroma 做到 STFT 上，得到时变 chromagram，调用 aurora.music |